# Train SOKE ASL 21K <=8s on Google Colab

Notebook nay dung theo flow hybrid: archive nam tren Google Drive, data/assets duoc copy va giai nen vao local `/content/SOKE_COLAB_DATA`, checkpoint ghi local truoc roi sync sang Drive moi epoch.

In [ ]:
# Sua 3 bien nay truoc khi chay.
GITHUB_REPO_URL = "https://github.com/nfisshy/SOKE-Speech-to-SignLanguage-Realtim.git"
GITHUB_BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/SOKE_COLAB"
LOCAL_ROOT = "/content/SOKE_COLAB_DATA"
LOCAL_ARCHIVE_ROOT = "/content/SOKE_COLAB_ARCHIVES"
LOCAL_RUN_ROOT = "/content/SOKE_COLAB_RUN"

REPO_DIR = "/content/SOKE"
CONFIG = "configs/soke_colab_asl_21k_8s.yaml"
ASSETS_CONFIG = "configs/assets_colab.yaml"

# Cach train cho dataset 21k_8s:
# - "finetune": load weight tu Drive last.ckpt, reset optimizer/scheduler, train run moi den TARGET_END_EPOCH.
# - "resume_full": resume ca optimizer/scheduler/trainer state, phu hop neu runtime bi ngat giua chung.
# - "fresh": train moi tu tokenizer.ckpt, khong dung last.ckpt.
TRAIN_MODE = "fresh"
TARGET_END_EPOCH = 80
BACKUP_BEFORE_FINETUNE = True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone", "--branch", GITHUB_BRANCH,
    GITHUB_REPO_URL, REPO_DIR
], check=True)
os.chdir(REPO_DIR)
print("Repo dir:", os.getcwd())

# Colab/PyTorch 2.6+ checkpoint compatibility hotfix.
# Safe for trusted local checkpoints: tokenizer.ckpt and your trained last.ckpt.
load_checkpoint_path = Path("mGPT/utils/load_checkpoint.py")
load_checkpoint_path.write_text('import torch\nfrom mGPT.utils.misc import neq_load_customized\n\n\ndef torch_load_trusted(path, map_location="cpu"):\n    try:\n        return torch.load(path, map_location=map_location, weights_only=False)\n    except TypeError:\n        return torch.load(path, map_location=map_location)\n\n\ndef load_pretrained(cfg, model, logger=None, phase="train"):\n    if phase == "train":\n        ckpt_path = cfg.TRAIN.PRETRAINED\n    elif phase == "test":\n        ckpt_path = cfg.TEST.CHECKPOINTS\n    else:\n        raise ValueError(f"Unsupported phase: {phase}")\n\n    if logger is not None:\n        logger.info(f"Loading pretrain model from {ckpt_path}")\n\n    state_dict = torch_load_trusted(ckpt_path, map_location="cpu")["state_dict"]\n    model.load_state_dict(state_dict, strict=False)\n    return model\n\n\ndef load_pretrained_vae(cfg, model, logger=None):\n    state_dict = torch_load_trusted(cfg.TRAIN.PRETRAINED_VAE, map_location="cpu")["state_dict"]\n    if logger is not None:\n        logger.info(f"Loading pretrain vae from {cfg.TRAIN.PRETRAINED_VAE}")\n\n    from collections import OrderedDict\n    vae_dict = OrderedDict()\n    hand_vae_dict = OrderedDict()\n    rhand_vae_dict = OrderedDict()\n    for k, v in state_dict.items():\n        if "motion_vae" in k:\n            vae_dict[k.replace("motion_vae.", "")] = v\n        elif "rhand_vae" in k:\n            rhand_vae_dict[k.replace("rhand_vae.", "")] = v\n        elif "hand_vae" in k:\n            hand_vae_dict[k.replace("hand_vae.", "")] = v\n        elif "vae" in k:\n            vae_dict[k.replace("vae.", "")] = v\n\n    if hasattr(model, "rhand_vae"):\n        print("load rhand vae...")\n        neq_load_customized(model.rhand_vae, rhand_vae_dict, verbose=True)\n    if hasattr(model, "hand_vae"):\n        print("load hand vae...")\n        neq_load_customized(model.hand_vae, hand_vae_dict, verbose=True)\n    if hasattr(model, "vae"):\n        print("load vae...")\n        neq_load_customized(model.vae, vae_dict, verbose=True)\n    else:\n        neq_load_customized(model.motion_vae, vae_dict, verbose=True)\n\n    return model\n', encoding="utf-8")
print("Patched:", load_checkpoint_path)


# Colab bf16 validation metric hotfix.
# SMPL hand regressors are float32; cast mesh vertices back to float32 before metric matmul.
t2m_metric_path = Path("mGPT/metrics/t2m.py")
t2m_metric_text = t2m_metric_path.read_text(encoding="utf-8")
t2m_metric_text = t2m_metric_text.replace(
    "vertices_rst = vertices_rst.detach().cpu()",
    "vertices_rst = vertices_rst.detach().cpu().float()",
)
t2m_metric_text = t2m_metric_text.replace(
    "vertices_ref = vertices_ref.detach().cpu()",
    "vertices_ref = vertices_ref.detach().cpu().float()",
)
t2m_metric_path.write_text(t2m_metric_text, encoding="utf-8")
print("Patched:", t2m_metric_path)


In [ ]:
import sys
import subprocess

# Khong cai requirements.txt goc vi co bpy va mot so package render khong can cho train.
# Tach theo nhom de neu fail se biet ngay nhom nao loi.
commands = [
    [sys.executable, "-m", "pip", "install", "-q", "-U", "pip", "setuptools", "wheel"],
    [sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "numpy==1.26.4"],
    [
        sys.executable, "-m", "pip", "install", "-q",
        "pytorch-lightning==2.4.0",
        "torchmetrics==1.4.3",
        "omegaconf==2.3.0",
        "shortuuid",
        "transformers==4.44.2",
        "sentencepiece",
        "tokenizers",
        "tqdm==4.67.0",
        "rich",
    ],
    [
        sys.executable, "-m", "pip", "install", "-q",
        "smplx==0.1.28",
        "trimesh==3.9.24",
        "pandas",
        "matplotlib",
        "spacy",
        "ftfy",
        "more-itertools",
        "natsort",
        "einops",
        "joblib==1.2.0",
        "h5py",
        "scikit-image",
        "wandb",
        "gdown",
    ],
]

def run(cmd):
    print("RUN:", " ".join(cmd))
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.returncode != 0:
        raise RuntimeError(f"Install failed: {' '.join(cmd)}")

for cmd in commands:
    run(cmd)

print("Dependency install done")
print("If the next cell raises numpy.dtype size changed, restart runtime once and run from the top.")


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

drive_root = Path(DRIVE_ROOT)
drive_archive_root = drive_root / "archives"
drive_token_cache = drive_root / "token_cache" / "TOKENS_how2sign_colab_21k_8s"
local_root = Path(LOCAL_ROOT)
local_archive_root = Path(LOCAL_ARCHIVE_ROOT)
local_run_root = Path(LOCAL_RUN_ROOT)
data_root = local_root / "data"
deps_root = local_root / "deps"
pretrained_root = local_root / "pretrained"
for path in [local_root, local_archive_root, local_run_root, data_root, deps_root, pretrained_root, drive_root / "experiments", drive_root / "results", drive_token_cache]:
    path.mkdir(parents=True, exist_ok=True)

def first_existing(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None

def copy_archive_to_local(archive_names):
    drive_archive = first_existing(*[drive_archive_root / name for name in archive_names])
    if drive_archive is None:
        return None
    local_archive = local_archive_root / drive_archive.name
    if (not local_archive.exists()) or local_archive.stat().st_size != drive_archive.stat().st_size:
        print("Copy archive to local:", drive_archive, "->", local_archive)
        shutil.copy2(drive_archive, local_archive)
    else:
        print("Local archive cache OK:", local_archive)
    return local_archive

def unpack_if_needed(expected_path, parent_dir, archive_names):
    expected_path = Path(expected_path)
    if expected_path.exists():
        print("Found:", expected_path)
        return
    archive = copy_archive_to_local(archive_names)
    if archive is None:
        print("Archive not found for:", expected_path)
        return
    print("Extracting", archive, "->", parent_dir)
    Path(parent_dir).mkdir(parents=True, exist_ok=True)
    if archive.suffix == ".zip":
        shutil.unpack_archive(str(archive), str(parent_dir))
    elif archive.name.endswith((".tar.gz", ".tgz")):
        subprocess.run(["tar", "-xzf", str(archive), "-C", str(parent_dir)], check=True)
    else:
        raise ValueError(f"Unsupported archive format: {archive}")

unpack_if_needed(data_root / "How2Sign", data_root, ["How2Sign.zip", "How2Sign.tar.gz", "How2Sign.tgz"])
unpack_if_needed(data_root / "stats", data_root, ["stats.zip", "stats.tar.gz", "stats.tgz"])
unpack_if_needed(deps_root / "mbart-h2s-csl-phoenix", deps_root, ["mbart-h2s-csl-phoenix.zip", "mbart-h2s-csl-phoenix.tar.gz", "mbart-h2s-csl-phoenix.tgz"])
unpack_if_needed(deps_root / "smpl_models", deps_root, ["smpl_models.zip", "smpl_models.tar.gz", "smpl_models.tgz"])
unpack_if_needed(deps_root / "t2m" / "glove", deps_root, ["t2m.tar.gz", "t2m.tgz", "t2m.zip"])

# t2m.tar.gz co the giai nen thanh layout long nhau tuy vao archive.
# Metric evaluator trong repo mong doi: deps/t2m/t2m/text_mot_match/model/finest.tar
# Nen tao symlink/copy ve dung path neu file nam o nested path khac.
def ensure_t2m_evaluator_layout():
    expected = deps_root / "t2m" / "t2m" / "text_mot_match" / "model" / "finest.tar"
    if expected.exists():
        print("Found t2m evaluator:", expected)
        return

    candidates = sorted((deps_root / "t2m").rglob("finest.tar")) if (deps_root / "t2m").exists() else []
    print("Expected t2m evaluator:", expected)
    print("Exists:", expected.exists())
    print("Found finest.tar candidates:")
    for candidate in candidates:
        print(" -", candidate)

    text_candidates = [candidate for candidate in candidates if "text_mot_match" in str(candidate)]
    if not text_candidates:
        raise FileNotFoundError(
            "Missing t2m text_mot_match/model/finest.tar. "
            "Hay upload t2m.tar.gz vao Drive archives va chay lai asset cell."
        )

    source = text_candidates[0]
    expected.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(source, expected)
        print("Symlink t2m evaluator:", expected, "->", source)
    except OSError:
        shutil.copy2(source, expected)
        print("Copied t2m evaluator:", source, "->", expected)

ensure_t2m_evaluator_layout()

tokenizer_from_archives = copy_archive_to_local(["tokenizer.ckpt"])
tokenizer_target = pretrained_root / "tokenizer.ckpt"
if tokenizer_from_archives and tokenizer_from_archives.exists():
    if (not tokenizer_target.exists()) or tokenizer_target.stat().st_size != tokenizer_from_archives.stat().st_size:
        shutil.copy2(tokenizer_from_archives, tokenizer_target)
        print("Copied tokenizer checkpoint to:", tokenizer_target)

required_paths = [
    data_root / "How2Sign" / "train" / "poses",
    data_root / "How2Sign" / "train" / "re_aligned" / "how2sign_realigned_train_preprocessed_fps.csv",
    data_root / "How2Sign" / "val" / "poses",
    data_root / "How2Sign" / "val" / "re_aligned" / "how2sign_realigned_val_preprocessed_fps.csv",
    data_root / "stats" / "mean.pt",
    data_root / "stats" / "std.pt",
    pretrained_root / "tokenizer.ckpt",
    deps_root / "mbart-h2s-csl-phoenix" / "map_ids.pkl",
    deps_root / "mbart-h2s-csl-phoenix" / "pytorch_model.bin",
    deps_root / "smpl_models" / "smplx" / "SMPLX_NEUTRAL.npz",
    deps_root / "smpl_models" / "smplx" / "SMPLX_to_J14.pkl",
    deps_root / "smpl_models" / "smplx_vert_segmentation.json",
    deps_root / "t2m" / "glove" / "our_vab_data.npy",
    deps_root / "t2m" / "t2m" / "text_mot_match" / "model" / "finest.tar",
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required Colab files after local extraction:\n" + "\n".join(missing))

Path("deps").mkdir(exist_ok=True)

def force_symlink(src, dst):
    dst = Path(dst)
    if dst.is_symlink() or dst.exists():
        if dst.is_dir() and not dst.is_symlink():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    os.symlink(src, dst, target_is_directory=True)

force_symlink(deps_root / "mbart-h2s-csl-phoenix", "deps/mbart-h2s-csl-phoenix")
force_symlink(deps_root / "smpl_models", "deps/smpl_models")
force_symlink(deps_root / "t2m", "deps/t2m")

print("Hybrid local data layout OK")

In [ ]:
import ast
import csv
import re
import sys
import shutil
import subprocess
from pathlib import Path

from omegaconf import OmegaConf

runtime_cfg = OmegaConf.load(CONFIG)
code_path = str(runtime_cfg.DATASET.CODE_PATH)
local_code_root = Path(LOCAL_ROOT) / "data" / "How2Sign" / code_path
local_token_dir = local_code_root / "how2sign"
drive_code_root = Path(DRIVE_ROOT) / "token_cache" / code_path
drive_token_dir = drive_code_root / "how2sign"
train_csv = Path(LOCAL_ROOT) / "data" / "How2Sign" / "train" / "re_aligned" / "how2sign_realigned_train_preprocessed_fps.csv"


def load_bad_how2sign_ids():
    dataset_file = Path(REPO_DIR) / "mGPT" / "data" / "humanml" / "dataset_t2m.py"
    text = dataset_file.read_text(encoding="utf-8")
    match = re.search(r"bad_how2sign_ids\s*=\s*(\[.*?\])", text)
    if not match:
        return set()
    return set(ast.literal_eval(match.group(1)))


def optional_positive_int(value):
    if value in (None, False, ""):
        return None
    value = int(value)
    return value if value > 0 else None


def expected_train_tokens():
    if not train_csv.exists():
        raise FileNotFoundError(f"Missing train CSV: {train_csv}")

    filter_cfg = runtime_cfg.DATASET.H2S.get("FILTER", {})
    max_duration = filter_cfg.get("TRAIN_MAX_DURATION", None)
    max_samples = optional_positive_int(filter_cfg.get("TRAIN_MAX_SAMPLES", None))
    bad_ids = load_bad_how2sign_ids()

    rows = []
    with train_csv.open("r", newline="", encoding="utf-8-sig") as file:
        reader = csv.DictReader(file)
        for row in reader:
            name = row["SENTENCE_NAME"]
            if name in bad_ids:
                continue
            if "DURATION" in row and row["DURATION"] not in (None, ""):
                duration = float(row["DURATION"])
            else:
                duration = float(row["END_REALIGNED"]) - float(row["START_REALIGNED"])
            if max_duration is not None and duration > float(max_duration):
                continue
            rows.append(row)

    if max_samples is not None and len(rows) > max_samples:
        rows = rows[:max_samples]
    return len(rows)


def count_tokens(path):
    return len(list(path.glob("*.npy"))) if path.exists() else 0


expected_count = expected_train_tokens()
local_token_count = count_tokens(local_token_dir)
drive_token_count = count_tokens(drive_token_dir)
print("Expected train token files:", expected_count)
print("Local token files:", local_token_count)
print("Drive token cache files:", drive_token_count)

if local_token_count < expected_count and drive_token_count >= expected_count:
    print("Restoring token cache from Drive to local disk")
    shutil.copytree(drive_code_root, local_code_root, dirs_exist_ok=True)
    local_token_count = count_tokens(local_token_dir)

if local_token_count < expected_count:
    cmd = [
        sys.executable, "-u", "-m", "scripts.get_motion_code",
        "--cfg", CONFIG,
        "--cfg_assets", ASSETS_CONFIG,
        "--nodebug",
        "--device", "0",
        "--use_gpus", "0",
    ]
    log_path = Path("/content/tokenize_debug.log")
    print("RUN:", " ".join(cmd))
    print("Log:", log_path)
    with log_path.open("w", encoding="utf-8", errors="replace") as log_file:
        process = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            log_file.write(line)
        return_code = process.wait()
    if return_code != 0:
        lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        print("".join(lines[-120:]))
        raise RuntimeError(f"get_motion_code failed with exit code {return_code}. Full log: {log_path}")
    local_token_count = count_tokens(local_token_dir)

if local_token_count < expected_count:
    raise RuntimeError(f"Token cache is incomplete: {local_token_count}/{expected_count} files in {local_token_dir}")

if count_tokens(drive_token_dir) < local_token_count:
    print("Saving local token cache back to Drive")
    drive_code_root.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_code_root, drive_code_root, dirs_exist_ok=True)
else:
    print("Token cache already exists; skip tokenization.")


In [ ]:
import sys
import shutil
import subprocess
from pathlib import Path
from omegaconf import OmegaConf

local_exp_dir = Path(LOCAL_RUN_ROOT) / "experiments" / "mgpt" / "SOKE_COLAB_ASL_21K_8S"
local_ckpt_dir = local_exp_dir / "checkpoints"
local_last_ckpt = local_ckpt_dir / "last.ckpt"
drive_exp_dir = Path(DRIVE_ROOT) / "experiments" / "mgpt" / "SOKE_COLAB_ASL_21K_8S"
drive_ckpt_dir = drive_exp_dir / "checkpoints"
drive_last_ckpt = drive_ckpt_dir / "last.ckpt"

train_mode = TRAIN_MODE.lower().strip()
if train_mode not in {"finetune", "resume_full", "fresh"}:
    raise ValueError(f"Unsupported TRAIN_MODE: {TRAIN_MODE}")

if train_mode in {"finetune", "resume_full"} and drive_last_ckpt.exists():
    local_ckpt_dir.mkdir(parents=True, exist_ok=True)
    should_restore = (
        (not local_last_ckpt.exists())
        or local_last_ckpt.stat().st_size != drive_last_ckpt.stat().st_size
        or drive_last_ckpt.stat().st_mtime > local_last_ckpt.stat().st_mtime
    )
    if should_restore:
        print("Restore checkpoint from Drive to local:", drive_last_ckpt, "->", local_last_ckpt)
        shutil.copy2(drive_last_ckpt, local_last_ckpt)
    else:
        print("Keep local checkpoint because it is newer or equal:", local_last_ckpt)

runtime_cfg_path = Path("/content/soke_colab_asl_21k_8s_runtime.yaml")
runtime_cfg = OmegaConf.load(CONFIG)
runtime_cfg.TRAIN.END_EPOCH = int(TARGET_END_EPOCH)
runtime_cfg.TRAIN.LR_SCHEDULER.params.T_max = int(TARGET_END_EPOCH)
runtime_cfg.TRAIN.RESUME = ""
runtime_cfg.TRAIN.PRETRAINED = ""
runtime_cfg.CHECKPOINT.SYNC_DIRPATH = str(drive_ckpt_dir)

if train_mode == "finetune":
    if not local_last_ckpt.exists():
        raise FileNotFoundError(
            "TRAIN_MODE='finetune' requires an existing last.ckpt in Drive/local. "
            f"Expected: {drive_last_ckpt} or {local_last_ckpt}"
        )
    if BACKUP_BEFORE_FINETUNE and drive_last_ckpt.exists():
        backup_ckpt = drive_ckpt_dir / "last_before_finetune_21k_8s.ckpt"
        if not backup_ckpt.exists():
            print("Backup current Drive last.ckpt before fine-tune:", backup_ckpt)
            shutil.copy2(drive_last_ckpt, backup_ckpt)
        else:
            print("Fine-tune backup already exists:", backup_ckpt)

    finetune_ckpt = Path(LOCAL_ROOT) / "pretrained" / "soke_asl_21k_8s_last_for_finetune.ckpt"
    finetune_ckpt.parent.mkdir(parents=True, exist_ok=True)
    if (not finetune_ckpt.exists()) or finetune_ckpt.stat().st_size != local_last_ckpt.stat().st_size:
        print("Copy fine-tune source checkpoint:", local_last_ckpt, "->", finetune_ckpt)
        shutil.copy2(local_last_ckpt, finetune_ckpt)
    runtime_cfg.TRAIN.PRETRAINED = str(finetune_ckpt)
    print("Fine-tune weights only; optimizer/scheduler will start fresh.")

elif train_mode == "resume_full":
    if not local_last_ckpt.exists():
        raise FileNotFoundError(
            "TRAIN_MODE='resume_full' requires local checkpoint. "
            f"Expected: {local_last_ckpt}"
        )
    print("Resume full trainer state from:", local_last_ckpt)

else:
    print("Start a fresh training run from tokenizer.ckpt only.")

OmegaConf.save(runtime_cfg, runtime_cfg_path)
print("Runtime config:", runtime_cfg_path)
print("TRAIN_MODE:", train_mode)
print("TARGET_END_EPOCH:", runtime_cfg.TRAIN.END_EPOCH)
print("TRAIN.PRETRAINED:", runtime_cfg.TRAIN.PRETRAINED)
print("CHECKPOINT.SYNC_DIRPATH:", runtime_cfg.CHECKPOINT.SYNC_DIRPATH)

cmd = [
    sys.executable, "-u", "-m", "train",
    "--cfg", str(runtime_cfg_path),
    "--cfg_assets", ASSETS_CONFIG,
    "--nodebug",
    "--device", "0",
    "--use_gpus", "0",
]

if train_mode == "resume_full":
    cmd += ["--resume", str(local_exp_dir)]

log_path = Path("/content/train_debug.log")
print("RUN:", " ".join(cmd))
print("Log:", log_path)
with log_path.open("w", encoding="utf-8", errors="replace") as log_file:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        log_file.write(line)
    return_code = process.wait()

if return_code != 0:
    print("\nTrain failed. Last log lines:")
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(lines[-200:]))
    raise RuntimeError(f"train.py failed with exit code {return_code}. Full log: {log_path}")


In [ ]:
from pathlib import Path

local_ckpt_dir = Path(LOCAL_RUN_ROOT) / "experiments" / "mgpt" / "SOKE_COLAB_ASL_21K_8S" / "checkpoints"
drive_ckpt_dir = Path(DRIVE_ROOT) / "experiments" / "mgpt" / "SOKE_COLAB_ASL_21K_8S" / "checkpoints"

print("Local checkpoint dir:", local_ckpt_dir)
for path in sorted(local_ckpt_dir.glob("*.ckpt")):
    print("local", path.name, f"{path.stat().st_size / (1024 ** 3):.2f} GiB")

print("Drive checkpoint dir:", drive_ckpt_dir)
for path in sorted(drive_ckpt_dir.glob("*.ckpt")):
    print("drive", path.name, f"{path.stat().st_size / (1024 ** 3):.2f} GiB")